# 04 — Training: DSen2 20m→10m super-resolution

Trains the PyTorch DSen2 network (`s2sr.model.DSen2Net20m`) on the patches extracted in `03_patch_extraction.ipynb`, using Wald's protocol to build synthetic (input, target) pairs from those native-resolution patches (`s2sr.dataset.WaldPairDataset`) — see `agents.md` "Methodology" and "Patch sampling strategy" for the reasoning.

Designed to run in two modes without any code changes, only the constants in the next cell:
- **Local smoke test** (CPU or a small local GPU): small subset of scenes, few epochs — just confirms the pipeline (data loading, shapes, loss going down) actually works before committing GPU time elsewhere.
- **Full run on RAILS** (A100): all scenes, more epochs, larger batch size, mixed precision.

This is a from-scratch reimplementation, not a port of DSen2's pretrained weights (GPL-3.0 licensing — see agents.md "Scope decision").

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

sys.path.insert(0, str(Path.cwd().parent / "src"))

from s2sr import config, raster
from s2sr.dataset import WaldPairDataset, train_val_split_by_scene
from s2sr.metrics import psnr, sam
from s2sr.model import DSen2Net20m

In [ ]:
# Flip these for a full run (RAILS/A100) vs a quick local smoke test.
SMOKE_TEST = True
MAX_SCENES = 5 if SMOKE_TEST else None  # None = use all scenes
NUM_EPOCHS = 3 if SMOKE_TEST else 50
BATCH_SIZE = 8 if SMOKE_TEST else 64
# Windows + Jupyter multiprocessing DataLoader workers can hang without an `if __name__ ==
# "__main__"` guard, which a notebook doesn't have — keep this 0 locally on Windows. Safe to
# raise (e.g. 4-8) on RAILS (Linux).
NUM_WORKERS = 0
LEARNING_RATE = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

In [ ]:
manifest = pd.read_csv(config.REPO_ROOT / "data" / "processed" / "patches_manifest.csv")

if MAX_SCENES is not None:
    keep_scenes = manifest["scene_id"].drop_duplicates().head(MAX_SCENES)
    manifest = manifest[manifest["scene_id"].isin(keep_scenes)].reset_index(drop=True)

print(f"{len(manifest)} patches from {manifest['scene_id'].nunique()} scenes")

In [ ]:
train_manifest, val_manifest = train_val_split_by_scene(manifest, val_fraction=0.15)
print(
    f"Train: {len(train_manifest)} patches / {train_manifest['scene_id'].nunique()} scenes\n"
    f"Val:   {len(val_manifest)} patches / {val_manifest['scene_id'].nunique()} scenes"
)

## Sanity check: one Wald's-protocol pair

Before training, look at what the dataset actually produces. `upsampled` (the network's low-res input) should look blurrier than `target` (the real ground truth it's trying to reconstruct) but still show the same features — if they look identical, the degradation isn't doing anything; if they look unrelated, something's misaligned.

In [ ]:
preview_ds = WaldPairDataset(train_manifest)
guide, upsampled, target = preview_ds[0]
print(f"guide: {tuple(guide.shape)}  upsampled: {tuple(upsampled.shape)}  target: {tuple(target.shape)}")

band_idx = config.BANDS_20M.index("B11")
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(raster.percentile_stretch(guide[[2, 1, 0]].permute(1, 2, 0).numpy()))  # B04,B03,B02
axes[0].set_title("guide (10m RGB)")
axes[1].imshow(upsampled[band_idx], cmap="gray")
axes[1].set_title("upsampled input (B11)")
axes[2].imshow(target[band_idx], cmap="gray")
axes[2].set_title("target (B11)")
for ax in axes:
    ax.axis("off")
fig.tight_layout()

In [ ]:
train_ds = WaldPairDataset(train_manifest)
val_ds = WaldPairDataset(val_manifest)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
len(train_loader), len(val_loader)

In [ ]:
model = DSen2Net20m(
    guide_channels=len(config.BANDS_10M),
    target_channels=len(config.BANDS_20M),
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = torch.nn.L1Loss()
scaler = torch.amp.GradScaler(device.type, enabled=(device.type == "cuda"))

sum(p.numel() for p in model.parameters())

## Training loop

Mixed precision (`torch.amp`) is enabled automatically when `device` is CUDA and is a no-op on CPU, so this cell doesn't need to change between the local smoke test and a RAILS run. The best (lowest validation L1 loss) checkpoint is kept, not just the last epoch's.

In [ ]:
def run_epoch(loader, train: bool):
    model.train(train)
    total_loss, n_samples = 0.0, 0
    with torch.set_grad_enabled(train):
        for guide, upsampled, target in loader:
            guide, upsampled, target = guide.to(device), upsampled.to(device), target.to(device)
            with torch.amp.autocast(device.type, enabled=(device.type == "cuda")):
                pred = model(guide, upsampled)
                loss = loss_fn(pred, target)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * guide.size(0)
            n_samples += guide.size(0)
    return total_loss / n_samples


history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")
checkpoint_path = config.REPO_ROOT / "models" / "dsen2_20m.pt"
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

for epoch in tqdm(range(NUM_EPOCHS)):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"epoch {epoch + 1}/{NUM_EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({"model_state_dict": model.state_dict(), "val_loss": val_loss, "epoch": epoch}, checkpoint_path)

print(f"Best val_loss: {best_val_loss:.4f} -> saved to {checkpoint_path}")

In [ ]:
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("epoch")
plt.ylabel("L1 loss")
plt.legend()
plt.title("Training curve")

## Evaluation: PSNR / SAM, against a bicubic baseline

L1 loss alone doesn't say whether the network is actually *learning* anything useful — a residual-learning model that predicts a ~zero correction would just reproduce the bicubic-upsampled input and still show a "reasonable" loss. The real question is whether the trained model beats plain bicubic upsampling (`upsampled`, already computed by `WaldPairDataset` — used here directly as the baseline) on standard super-resolution metrics:

- **PSNR** (peak signal-to-noise ratio, dB): higher is better, sensitive to overall magnitude/brightness error.
- **SAM** (spectral angle mapper, degrees): lower is better, measures per-pixel spectral-*shape* fidelity independent of brightness — this is the one that matters most for a genuinely multi-band task like this.

If the model doesn't clearly beat the bicubic baseline on both, something's off (undertrained, learning rate, or a bug) — it should not be possible for a trained residual network to do *worse* than adding nothing.

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded checkpoint from epoch {checkpoint['epoch'] + 1}, val_loss={checkpoint['val_loss']:.4f}")


def evaluate(loader):
    model.eval()
    totals = {"pred_psnr": 0.0, "pred_sam": 0.0, "baseline_psnr": 0.0, "baseline_sam": 0.0}
    n = 0
    with torch.no_grad():
        for guide, upsampled, target in loader:
            guide, upsampled, target = guide.to(device), upsampled.to(device), target.to(device)
            pred = model(guide, upsampled)
            b = guide.size(0)
            totals["pred_psnr"] += psnr(pred, target).item() * b
            totals["pred_sam"] += sam(pred, target).item() * b
            totals["baseline_psnr"] += psnr(upsampled, target).item() * b
            totals["baseline_sam"] += sam(upsampled, target).item() * b
            n += b
    return {k: v / n for k, v in totals.items()}


val_metrics = evaluate(val_loader)
print(f"Model:            PSNR={val_metrics['pred_psnr']:.2f} dB   SAM={val_metrics['pred_sam']:.2f}°")
print(f"Bicubic baseline: PSNR={val_metrics['baseline_psnr']:.2f} dB   SAM={val_metrics['baseline_sam']:.2f}°")

In [ ]:
n_preview = 3
band_idx = config.BANDS_20M.index("B11")
fig, axes = plt.subplots(n_preview, 3, figsize=(9, 3 * n_preview))
model.eval()
with torch.no_grad():
    for i in range(n_preview):
        guide, upsampled, target = val_ds[i]
        pred = model(guide.unsqueeze(0).to(device), upsampled.unsqueeze(0).to(device))[0].cpu()
        for ax, img, title in zip(
            axes[i],
            [upsampled[band_idx], pred[band_idx], target[band_idx]],
            ["bicubic baseline", "model prediction", "ground truth"],
        ):
            ax.imshow(img, cmap="gray")
            ax.set_title(title, fontsize=9)
            ax.axis("off")
fig.tight_layout()

## Next steps

- If this was a smoke test (`SMOKE_TEST = True`): confirm the loss actually decreased and nothing errored, then flip `SMOKE_TEST = False` and re-run on RAILS with the full patch set.
- Check the PSNR/SAM cells above: the model should clearly beat the bicubic baseline on both. If it doesn't, that points to undertraining, a learning-rate issue, or a bug — not a fundamental limit.
- Currently only the 20m→10m branch is implemented — B01/B09 (60m) are not extracted by notebook 3 at all (see `s2sr.patches`); revisit if the 60m bands turn out to be needed downstream.
- Land-cover stratification (still not implemented — see agents.md open caveat) would be worth revisiting if validation performance looks skewed toward whichever land cover is spatially dominant.